# TradeFlowGNN: Data Exploration

This notebook explores the preprocessed CEPII Gravity dataset and visualizes the global trade network.

In [ ]:
# ── Colab/Notebook Setup ───────────────────────────────────────────
import sys
from pathlib import Path
import os

def setup_colab():
    if 'google.colab' in str(get_ipython()):
        print("Running in Google Colab. Setting up environment...")
        
        # 1. Install dependencies
        !pip install -q pyarrow pandas numpy matplotlib seaborn tqdm pyyaml
        
        # 2. Clone repository if not already present
        repo_name = "TradeFlowGCN"
        if not Path(repo_name).exists() and not Path("src").exists():
            !git clone https://github.com/tolyaho/TradeFlowGCN.git
        
        # 3. Enter repository directory
        if Path(repo_name).exists() and not Path("src").exists():
            os.chdir(repo_name)
            print(f"Changed directory to {os.getcwd()}")
        
        # 4. Set PYTHONPATH
        sys.path.append(os.getcwd())
        sys.path.append(os.path.join(os.getcwd(), "src"))

setup_colab()

# ── Robust Path Setup ──────────────────────────────────────────────
curr_path = Path(".").resolve()
root_path = curr_path
while not (root_path / "src").exists() and root_path.parent != root_path:
    root_path = root_path.parent

if (root_path / "src").exists():
    if str(root_path / "src") not in sys.path:
        sys.path.append(str(root_path / "src"))
    print(f"Project Root: {root_path}")
    print(f"Added to sys.path: {root_path / 'src'}")
else:
    print(f"Warning: Could not find 'src' directory relative to {curr_path}")
    print("If in Colab, ensure you have cloned the repo and are in the correct directory.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

## 1. Load Data

We'll look for the latest cached parquet file in `data/processed/`.

In [ ]:
processed_dir = root_path / "data/processed"
parquet_files = list(processed_dir.glob("*.parquet"))

if not parquet_files:
    print(f"No cached data found in {processed_dir}. Please run `uv run python scripts/train.py` first.")
else:
    latest_file = sorted(parquet_files)[-1]
    print(f"Loading: {latest_file}")
    df = pd.read_parquet(latest_file)
    display(df.head())

## 2. Basic Statistics

In [ ]:
print(f"Total rows: {len(df)}")
print(f"Year range: {df['year'].min()} to {df['year'].max()}")
print(f"Unique countries: {len(df['iso3_o'].unique())}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['log_trade'], bins=50, kde=True)
plt.title("Distribution of Log-Trade Values")
plt.xlabel("log(1 + tradeflow)")
plt.show()

## 3. Top Trading Pairs (Latest Year)

In [ ]:
latest_year = df['year'].max()
top_pairs = df[df['year'] == latest_year].sort_values('tradeflow_comtrade_o', ascending=False).head(10)
top_pairs[['iso3_o', 'iso3_d', 'tradeflow_comtrade_o']]